# Урок 5.2. Выбор модели под задачу: не параметры, а измеримые метрики

Модуль 5 «Локальные LLM: запуск без облака и без GPU» · урок 2 из 7

Урок 5.1 закончился правилом: локальная модель имеет смысл, только если доказано,
что она решает вашу задачу. Этот урок — про то, как это доказывать: сначала физика
(что влезет в RAM и с какой скоростью поедет), потом измерения (мини-бенчмарк
на своих задачах). Все замеры в уроке — живые, с моей машины; у вас числа будут
свои, и это нормально — важен метод.

## 🎯 Что вы узнаете

- Что такое квантизация и формат GGUF, чем Q4_K_M отличается от Q8_0 — и на что это влияет
- Сколько RAM реально нужно моделям 1B–8B в разных квантизациях (таблица + формула)
- Почему скорость генерации на CPU упирается в пропускную способность памяти — и как проверить формулу живым замером
- Почему публичным лидербордам нельзя доверять выбор модели
- Как собрать собственный мини-бенчмарк на своих задачах и выбирать модель по цифрам


## Что означает «7B» и почему модель не влезает в память

«7B» — это 7 миллиардов параметров (parameters): обучаемых чисел-весов. В исходном виде каждый вес хранится как 16-битное число с плавающей точкой (FP16/BF16), то есть **2 байта на вес**:

```
7B параметров x 2 байта = ~14 ГБ  — только веса, без учёта всего остального
```

Для ноутбука с 16 ГБ RAM это приговор. Решение — **квантизация (quantization)**: хранить веса с меньшей точностью.

Аналогия для backend-разработчика: у вас колонка `DECIMAL(38, 18)`, а бизнесу хватает `DECIMAL(10, 2)` — вы меняете тип, таблица худеет вчетверо, а отчёты сходятся с точностью до копейки. Или JPEG: качество 85 неотличимо от оригинала на глаз, а файл в 5 раз меньше. Квантизация LLM — то же самое: немного жертвуем точностью весов, кратно выигрываем в памяти и скорости.

## GGUF: формат, вокруг которого всё крутится

**GGUF** (GPT-Generated Unified Format) — бинарный формат файла модели для движка llama.cpp (и всего, что на нём построено, включая Ollama). Один файл содержит: квантизованные веса + токенизатор + метаданные (архитектура, длина контекста, шаблон чата). Скачал один файл — модель готова к запуску. Это как fat JAR в мире Java или самодостаточный Docker-образ.

Названия квантизаций выглядят так: `Q4_K_M`, `Q5_K_S`, `Q8_0`. Расшифровка:

- `Q4` — примерно 4 бита на вес (вместо 16);
- `K` — семейство «k-quants»: веса группируются в блоки, у каждого блока свой масштаб — точнее, чем старое линейное квантование;
- `S / M / L` (small/medium/large) — насколько агрессивно квантованы отдельные «чувствительные» слои; M — сбалансированный выбор.

| Квантизация | Бит/вес (≈) | Размер относительно FP16 | Потеря качества | Когда брать |
|---|---|---|---|---|
| Q2_K, Q3_K | 2.5–3.5 | ~18–25% | Заметная, модель «глупеет» | Только если совсем не влезает; лучше взять модель поменьше в Q4 |
| **Q4_K_M** | ~4.8 | ~30% | Небольшая | **Дефолт для CPU: лучший баланс** (его Ollama качает по умолчанию) |
| Q5_K_M | ~5.7 | ~36% | Едва заметная | Если RAM позволяет, а задача чувствительна к качеству |
| Q8_0 | ~8.5 | ~53% | Практически нет | Эталон для сравнения, критичные задачи; вдвое тяжелее и медленнее Q4 |
| F16 (без квантизации) | 16 | 100% | — | На CPU почти никогда: медленно и прожорливо |

Практическое правило: **большая модель в Q4 почти всегда лучше маленькой в Q8** при одинаковом объёме памяти. 8B-Q4 (≈4.9 ГБ) обычно умнее, чем 3B-Q8 (≈3.4 ГБ).

## Сколько нужно RAM: таблица и формула

Память при работе складывается из трёх частей:

```
RAM = вес файла модели  +  KV-кеш (растёт с контекстом)  +  ~0.5-1 ГБ (runtime, буферы)
```

**KV-кеш** — это «память диалога» модели: кеш промежуточных значений для каждого токена контекста, чтобы не пересчитывать их заново (прямой аналог материализованного представления или кеша вычислений). Растёт линейно с длиной контекста; для типичной 8B-модели это ~128 КБ на токен → 4096 токенов ≈ 0.5 ГБ. Подробная формула будет в уроке 5.6.

Размеры файлов GGUF популярных моделей (округлённо, у конкретных релизов ±10%):

| Модель | Параметры | Q4_K_M | Q5_K_M | Q8_0 | F16 |
|---|---|---|---|---|---|
| Llama 3.2 1B | 1.2B | 0.8 ГБ | 0.9 ГБ | 1.3 ГБ | 2.5 ГБ |
| Llama 3.2 3B | 3.2B | 2.0 ГБ | 2.3 ГБ | 3.4 ГБ | 6.4 ГБ |
| Phi-4-mini | 3.8B | 2.4 ГБ | 2.8 ГБ | 4.1 ГБ | 7.7 ГБ |
| Qwen2.5 7B | 7.6B | 4.7 ГБ | 5.4 ГБ | 8.1 ГБ | 15.2 ГБ |
| Llama 3.1 8B | 8.0B | 4.9 ГБ | 5.7 ГБ | 8.5 ГБ | 16.1 ГБ |

Итого **рабочая RAM при контексте 4k**: 1B-Q4 ≈ 1.5–2 ГБ · 3B-Q4 ≈ 3–3.5 ГБ · 8B-Q4 ≈ 6–7 ГБ. Плюс не забывайте: ОС + браузер съедают свои 3–4 ГБ.

| Ваша RAM | Что реально запускать |
|---|---|
| 8 ГБ | 1B–3B в Q4; 7B — только закрыв всё остальное, на грани swap |
| 16 ГБ | 7–8B в Q4/Q5 комфортно; 13–14B в Q4 — на пределе |
| 32 ГБ | 14B в Q5, эксперименты с 27–32B в Q4 (но скорость на CPU будет 1–3 ток/с) |

## Скорость на CPU: физика, а не магия

Ключевой факт, который объясняет все цифры ниже: **генерация упирается в пропускную
способность памяти (memory bandwidth), а не в мощность ядер**. Чтобы предсказать один
токен, процессор должен прочитать из RAM все веса модели — целиком, каждый раз.

```
скорость генерации (ток/с) <= пропускная способность RAM / размер модели в байтах

DDR4 dual-channel: ~50 ГБ/с (пик)      DDR5 dual-channel: ~75-90 ГБ/с (пик)

3B в Q4_K_M = ~1.8 ГБ  ->  DDR5-4800: 76.8 / 1.8 ~= 42 ток/с (теоретический ПОТОЛОК)
```

Потолок — это именно потолок: реальный КПД по нашим замерам — **30–60% от пиковой
полосы** (дековантизация весов тоже ест такты, ноутбуки троттлят, часть полосы
съедает система). Ниже мы проверим формулу живым замером — спойлер: мой ноутбук
выдал ~40% от пика.

Честные ориентиры (генерация, короткий контекст, Q4_K_M):

| Модель | Ноутбук (DDR4 / троттлящий DDR5) | Десктоп (DDR5, без троттлинга) |
|---|---|---|
| 1B | 20–45 ток/с | 45–80 ток/с |
| 3B | 8–20 ток/с | 20–35 ток/с |
| 7–8B | 4–9 ток/с | 8–16 ток/с |
| 13–14B | 2–5 ток/с | 4–8 ток/с |

Для ориентира: комфортное чтение — 5–10 ток/с, то есть 7–8B на CPU «читается»
на грани комфорта, а 1–3B летают.

Два важных следствия:

1. **Больше ядер ≠ быстрее генерация.** Ядра помогают на этапе обработки промпта
   (prefill) — он считается батчем и параллелится. Но чудес не ждите: на CPU честный
   prefill быстрее генерации **в разы, а не в десятки раз** (замер ниже: ~50–60
   против 15–17 ток/с; в десятки раз — только на GPU). Главный ускоритель длинных
   промптов на CPU — **кеш префикса**, и ниже мы поймаем его с поличным.
2. **Q8 вдвое медленнее Q4** — читать из RAM приходится вдвое больше байтов.
   Квантизация ускоряет, а не только экономит память.


Переведём формулы в код — калькулятор «влезет и с какой скоростью поедет».

In [ ]:
BITS_PER_WEIGHT = {  # эффективные биты на вес (с учётом служебных данных формата)
    "Q2_K": 3.35, "Q3_K_M": 3.9, "Q4_K_M": 4.85, "Q5_K_M": 5.7, "Q6_K": 6.6, "Q8_0": 8.5, "F16": 16.0,
}


def model_file_gb(params_b: float, quant: str) -> float:
    """Оценка размера GGUF-файла в ГБ: параметры (млрд) x биты / 8."""
    return params_b * 1e9 * BITS_PER_WEIGHT[quant] / 8 / 1024**3


def estimate_ram_gb(params_b: float, quant: str, ctx_tokens: int = 4096,
                    kv_kb_per_token: float = 128.0) -> float:
    """Рабочая RAM: файл + KV-кеш + ~0.7 ГБ на runtime.

    kv_kb_per_token: ~128 КБ/токен для 3-8B моделей с GQA, ~32 КБ для 1B.
    """
    kv_gb = ctx_tokens * kv_kb_per_token / 1024 / 1024
    return model_file_gb(params_b, quant) + kv_gb + 0.7


def estimate_speed_tps(params_b: float, quant: str, bandwidth_gb_s: float,
                       efficiency: float = 0.45) -> float:
    """Оценка скорости генерации: пиковая полоса RAM / размер модели * КПД.

    КПД 0.3-0.6 по живым замерам; пиковая полоса недостижима.
    """
    return bandwidth_gb_s / model_file_gb(params_b, quant) * efficiency


print(f"{'модель':<8} {'квант':<8} {'файл, ГБ':>9} {'RAM@4k, ГБ':>11} "
      f"{'DDR4-3200':>10} {'DDR5-4800':>10}")
print("-" * 62)
for name, params in [("1B", 1.2), ("3B", 3.2), ("8B", 8.0)]:
    for quant in ("Q4_K_M", "Q5_K_M", "Q8_0"):
        kv = 32.0 if params < 2 else 128.0
        print(f"{name:<8} {quant:<8} {model_file_gb(params, quant):>9.1f} "
              f"{estimate_ram_gb(params, quant, kv_kb_per_token=kv):>11.1f} "
              f"{estimate_speed_tps(params, quant, 51.2):>8.1f}/с "
              f"{estimate_speed_tps(params, quant, 76.8):>8.1f}/с")

## Проверяем формулу на себе

Формула красивая, но курс приучил нас не верить формулам без прогона. Мой стенд:
ноутбук, 2 модуля DDR5-4800 в двухканале — пиковая полоса `4800 МТ/с × 8 байт × 2
канала = 76.8 ГБ/с`. Файл qwen2.5:3b (Q4_K_M) — 1.9 ГБ. Прогноз по формуле:
потолок `76.8 / 1.9 ≈ 40 ток/с`, с КПД 0.45 — **~18 ток/с**.

Свою пиковую полосу посчитаете так же: PowerShell →
`Get-CimInstance Win32_PhysicalMemory | Select Speed, DeviceLocator` (скорость в МТ/с,
число строк = число модулей; два модуля в разных каналах = dual-channel).

Есть нюанс: в моём ноутбуке есть GPU, и Ollama по умолчанию кладёт модель туда.
Для честного CPU-замера принудим CPU опцией `num_gpu: 0` (ноль слоёв на GPU).
Заодно увидим, что даёт GPU, — а если у вас его нет, обе пары чисел просто совпадут.


In [ ]:
import httpx

# не localhost: на Windows httpx сперва пробует IPv6 ::1 и теряет ~1-2 с
# на каждом свежем соединении (грабля из урока 3.2), а Ollama слушает 127.0.0.1
OLLAMA_URL = "http://127.0.0.1:11434"
NS = 1_000_000_000  # Ollama отдаёт длительности в наносекундах


def ollama_available() -> bool:
    try:
        return httpx.get(f"{OLLAMA_URL}/api/version", timeout=2).status_code == 200
    except httpx.HTTPError:
        return False


OLLAMA_UP = ollama_available()
print("Ollama доступна" if OLLAMA_UP else
      "Ollama не запущена — ячейки с запросами будут пропущены. Запустите её и перезапустите ноутбук.")

In [ ]:
MODEL = "qwen3.5:9b"  # рабочая модель курса (скачана в уроке 1.5)
PROMPT_SPEED = ("Объясни своими словами, зачем веб-сервису нужна очередь сообщений "
                "и какие проблемы она решает. Ответь развёрнуто, 5-7 предложений.")


def measure_speed(model: str, *, force_cpu: bool = False, label: str = "") -> None:
    options = {"temperature": 0, "seed": 42, "num_predict": 200}
    if force_cpu:
        options["num_gpu"] = 0  # 0 слоёв на GPU = чистый CPU-инференс
    d = httpx.post(f"{OLLAMA_URL}/api/generate", json={
        "model": model, "prompt": PROMPT_SPEED, "stream": False, "options": options,
    }, timeout=600).raise_for_status().json()
    prefill_tps = d["prompt_eval_count"] / (d["prompt_eval_duration"] / NS)
    gen_tps = d["eval_count"] / (d["eval_duration"] / NS)
    print(f"{label:<24} prefill {prefill_tps:7.1f} ток/с | генерация {gen_tps:5.1f} ток/с | "
          f"загрузка {d.get('load_duration', 0) / NS:.1f} с")


if OLLAMA_UP:
    measure_speed(MODEL, label="как есть (у меня GPU)")
    measure_speed(MODEL, label="как есть, повтор")
    measure_speed(MODEL, force_cpu=True, label="принудительный CPU")
    measure_speed(MODEL, force_cpu=True, label="CPU, повтор (тёплый)")
    # Смена опций памяти (num_gpu) перезапустила runner с новым конфигом, и он
    # ОСТАНЕТСЯ в CPU-режиме даже для запросов без опций (подробнее — урок 5.3).
    # Выгружаем модель, чтобы вернуть поведение по умолчанию:
    httpx.post(f"{OLLAMA_URL}/api/generate", json={"model": MODEL, "keep_alive": 0}, timeout=60)
else:
    print("Пропущено: Ollama не запущена")

Мой прогон (ваши числа будут отличаться — важно соотношение):

| режим | генерация |
|---|---|
| GPU (по умолчанию у меня) | **~61 ток/с** |
| принудительный CPU | **15–17 ток/с** |

Что тут видно:

- **Формула работает, но с честным КПД.** Прогноз с КПД 0.45 давал ~18 ток/с,
  факт — 15–17 (в разных прогонах я видел и 14.6): реальный КПД моего ноутбука
  `16 × 1.9 / 76.8 ≈ 40%` от пиковой полосы. Троттлинг тонкого корпуса
  и накладные расходы деквантизации съели своё.
- Формула нужна не для точного прогноза, а чтобы **не ждать чуда**: 8B на этой же
  машине поедет ~16 × (1.9/4.5) ≈ 7 ток/с — на грани комфортного чтения.
- **GPU ускоряет генерацию в ~3.5–4 раза** на этой модели (61 против 15–17).
- В первом CPU-прогоне `загрузка` не нулевая: смена `num_gpu` заставила Ollama
  перезагрузить модель. Параметры памяти перезапускают runner — запомните это
  до урока 5.3.

А вот с **prefill** в этом замере творится что-то странное: между первым прогоном
и «повтором» скорость прыгает на порядок (в моих прогонах: 350–650 → 3700–5100 ток/с на GPU,
~65 → 1050–1150 на CPU). Модель не могла внезапно стать в 16 раз быстрее. Разберёмся.


### Prefill-детектив: кеш префикса

Гипотеза: оба прогона используют **один и тот же промпт**. Ollama (как и llama.cpp
под ней) хранит KV-кеш последнего запроса и, увидев совпадающее начало промпта,
**не пересчитывает его** — а `prompt_eval_count` в ответе всё равно рапортует полное
число токенов. Получается дутая «скорость»: токены засчитаны, а работы не было.

Это тот же эффект, что прогретый кеш запросов в БД: `EXPLAIN ANALYZE` на повторе
показывает милые миллисекунды, а первый холодный запрос шёл секунды.

Проверим гипотезу: сгенерируем промпты **без общего начала** (каждый начинается
со своего номера) и сравним со «своим повтором»:


In [ ]:
WORDS = ["сервер", "очередь", "кластер", "маршрут", "кеш", "индекс", "запрос",
         "токен", "буфер", "реплика", "шардинг", "бэкап", "монитор", "конвейер"]


def make_unique_prompt(seed: int, n_words: int = 250) -> str:
    """Промпты с РАЗНЫМ началом и разной перестановкой слов — общего префикса нет."""
    body = " ".join(WORDS[(seed + i) % len(WORDS)] for i in range(n_words))
    return f"Промпт номер {seed}. Сколько раз в списке встречается слово «кеш»? Список: {body}"


def probe_prefill(prompt: str, label: str, force_cpu: bool = False) -> None:
    options = {"temperature": 0, "seed": 42, "num_predict": 1}
    if force_cpu:
        options["num_gpu"] = 0
    d = httpx.post(f"{OLLAMA_URL}/api/generate", json={
        "model": MODEL, "prompt": prompt, "stream": False, "options": options,
    }, timeout=600).raise_for_status().json()
    pe_c, pe_d = d.get("prompt_eval_count", 0), d.get("prompt_eval_duration", 0)
    tps = pe_c / (pe_d / NS) if pe_d else 0
    print(f"  {label:<34} prompt_eval={pe_c:>4} ток @ {tps:8.1f} ток/с")


def unload_model() -> None:
    httpx.post(f"{OLLAMA_URL}/api/generate", json={"model": MODEL, "keep_alive": 0}, timeout=60)


if OLLAMA_UP:
    print("--- принудительный CPU ---")
    unload_model()
    probe_prefill(make_unique_prompt(1), "уникальный №1 (после загрузки)", True)
    probe_prefill(make_unique_prompt(2), "уникальный №2", True)
    probe_prefill(make_unique_prompt(2), "ПОВТОР №2 (кеш префикса!)", True)
    print("--- GPU (по умолчанию) ---")
    unload_model()
    probe_prefill(make_unique_prompt(3), "уникальный №3 (после загрузки)")
    probe_prefill(make_unique_prompt(4), "уникальный №4")
    unload_model()  # прибираем за собой
else:
    print("Пропущено: Ollama не запущена")

Мой вердикт (числа из прогона выше и из более длинного замера на ~1900 токенах):

| prefill | честный (уникальный промпт) | повтор того же промпта |
|---|---|---|
| CPU | **~50–66 ток/с** | тысячи «ток/с» (кеш) |
| GPU | **~2500–3100 ток/с** | десятки тысяч «ток/с» (кеш) |

Три вывода, каждый из которых стоит денег:

1. **Честный CPU-prefill лишь в ~3–4 раза быстрее генерации** (50–60 против 15–17),
   а не в десятки раз, как обещают популярные шпаргалки. 1500-токенный промпт
   «с нуля» на CPU — это ~25–30 секунд. На GPU — да, в десятки раз (~2500–3100 против 61).
2. **Кеш префикса — главный друг шаблонных промптов.** Типовой продовый промпт —
   это неизменные system + few-shot (сотни токенов) и короткий меняющийся хвост.
   Совпадающее начало не пересчитывается — повторные запросы платят только за хвост.
   Вот почему «длинный вход, короткий выход» из урока 5.1 остаётся лучшим профилем
   для CPU: длинная часть входа обычно шаблонная.
3. **Меряйте скорость уникальными промптами.** Бенчмарк, гоняющий один и тот же
   промпт, измеряет кеш, а не модель, — мы сами чуть не попались абзацем выше.


## Какие модели вообще смотреть (снимок на начало 2026)

Ландшафт меняется каждые несколько месяцев, поэтому запоминайте не названия, а **семейства и метод выбора**:

- **Llama 3.2 (1B, 3B) / Llama 3.1 (8B)** — Meta; ровные универсалы, огромная экосистема. Лицензия Llama (для большинства коммерческих задач ок, читайте условия).
- **Qwen 2.5 / Qwen 3 (0.5B–32B+)** — Alibaba; сильны в мультиязычности (включая русский), коде и JSON-дисциплине. Apache 2.0 у большинства размеров.
- **Phi-4 / Phi-4-mini (3.8B, 14B)** — Microsoft; «плотные знания» при малом размере, хороши в рассуждениях. MIT.
- **Gemma 2/3 (2B–27B)** — Google; аккуратные ответы, свои условия использования.
- **Mistral 7B / Ministral** — Mistral AI; быстрые, Apache 2.0.

В этом курсе всё работает на **qwen2.5:3b** — она уже скачана с урока 1.5, хороша в русском и дисциплинированна в JSON (мы убедились в этом в модулях 1–4).

Берите **instruct-версии** (дообученные следовать инструкциям), а не base: base-модель — это «автодополнялка текста», она не отвечает на вопросы, а продолжает их.

## Почему чужие бенчмарки не выбирают модель за вас

Публичные лидерборды (MMLU, Arena и т.п.) — это как TPC-бенчмарки для СУБД: полезно для общего ориентира, бесполезно для вашего конкретного workload. Причины:

1. **Контаминация**: тестовые наборы утекают в обучающие данные, модели «зубрят» ответы.
2. **Англоцентричность**: модель может блистать в английском MMLU и путаться в русских падежах.
3. **Ваша задача ≠ академический тест**: «извлеки ИНН из письма и верни JSON» нет ни в одном лидерборде.
4. **Формат важнее IQ**: в проде модель часто губит не «глупость», а неумение стабильно отдавать нужный формат.

Решение то же, что и в мире БД: **прогнать свой workload**. Собираем мини-бенчмарк: 10–30 маленьких задач из вашей реальной практики с автоматической проверкой. Это дёшево (полчаса работы), а окупается при каждом выборе и каждом обновлении модели — как регрессионные тесты.

## Практика: собственный мини-бенчмарк

План:

1. Задачи лежат в `data/benchmark_tasks.jsonl` — по одной на строку: промпт + тип проверки + ожидание.
2. Прогоняем каждую через модель в Ollama с `temperature=0` (детерминизм).
3. Проверяем ответ автоматической функцией-чекером.
4. Заодно снимаем скорость: Ollama возвращает `eval_count` (сколько токенов сгенерировано) и `eval_duration` (за сколько наносекунд).

Типы проверок — четыре, по возрастанию мягкости:

| check | Что проверяет | Пример задачи |
|---|---|---|
| `exact` | Точное совпадение (после нормализации) | классификация: positive/negative |
| `regex` | Совпадение с регулярным выражением | «ответь только числом» |
| `contains_all` | Все ключевые подстроки на месте | суммаризация с сохранением названия |
| `json_keys` | Ответ парсится как JSON и содержит ключи | структурированное извлечение |

Доступность Ollama мы уже проверили выше (`OLLAMA_UP`) — переходим к задачам и чекерам.

Загружаем задачи и пишем чекеры. Обратите внимание на `check_json_keys`: модели любят оборачивать JSON в пояснения или markdown-блоки, поэтому сначала вырезаем первый `{...}` из ответа — маленькая, но важная продовая привычка.

In [ ]:
import json
import re
from pathlib import Path

tasks = [
    json.loads(line)
    for line in Path("data/benchmark_tasks.jsonl").read_text(encoding="utf-8").splitlines()
    if line.strip()
]
print(f"Задач: {len(tasks)}; категории: {sorted({t['category'] for t in tasks})}")


def check_exact(answer: str, expected) -> bool:
    return answer.strip().strip(".!«»\"'").lower() == str(expected).lower()


def check_contains_all(answer: str, expected) -> bool:
    return all(part.lower() in answer.lower() for part in expected)


def check_regex(answer: str, expected) -> bool:
    return re.search(str(expected), answer.strip(), re.MULTILINE) is not None


def check_json_keys(answer: str, expected) -> bool:
    match = re.search(r"\{.*\}", answer, re.DOTALL)  # достаём первый {...}, игнорируя обвязку
    if not match:
        return False
    try:
        obj = json.loads(match.group(0))
    except json.JSONDecodeError:
        return False
    return isinstance(obj, dict) and all(key in obj for key in expected)


CHECKERS = {"exact": check_exact, "contains_all": check_contains_all,
            "regex": check_regex, "json_keys": check_json_keys}

Раннер: один запрос к `/api/chat` (без стриминга), фиксированный `seed` и `temperature=0` — чтобы прогоны были воспроизводимыми, как юнит-тесты. `num_predict` ограничивает длину ответа: для классификации незачем ждать сочинение.

In [ ]:
def run_task(model: str, task: dict, timeout: float = 300.0) -> dict:
    payload = {
        "model": model,
        "messages": [{"role": "user", "content": task["prompt"]}],
        "stream": False,
        "think": False,
        "options": {"temperature": 0, "seed": 42, "num_predict": task.get("max_tokens", 2000)},
    }
    resp = httpx.post(f"{OLLAMA_URL}/api/chat", json=payload, timeout=timeout)
    resp.raise_for_status()
    data = resp.json()
    answer = data["message"]["content"]
    eval_count, eval_duration = data.get("eval_count", 0), data.get("eval_duration", 0)
    return {
        "id": task["id"],
        "category": task["category"],
        "passed": CHECKERS[task["check"]](answer, task["expected"]),
        "answer": answer.strip(),
        "done_reason": data.get("done_reason", "?"),
        "tps": eval_count / (eval_duration / NS) if eval_duration else None,
    }


def bench(model: str, tasks: list[dict]) -> dict:
    results = [run_task(model, t) for t in tasks]
    speeds = [r["tps"] for r in results if r["tps"]]
    by_cat: dict[str, list[bool]] = {}
    for r in results:
        by_cat.setdefault(r["category"], []).append(r["passed"])
    return {
        "model": model,
        "score": sum(r["passed"] for r in results) / len(results),
        "median_tps": sorted(speeds)[len(speeds) // 2] if speeds else None,
        "by_category": {c: sum(v) / len(v) for c, v in sorted(by_cat.items())},
        "results": results,
    }

Запускаем на моделях, которые в курсе уже скачаны: **qwen2.5:3b** (наша рабочая
instruct-модель) и **qwen2.5:0.5b-base** (base-модель из урока 1.6 — сейчас увидите,
зачем она в этом списке). Если хотите честного соседа по размеру — скачайте
`ollama pull qwen2.5:1.5b` (~1 ГБ), раннер сам подхватит её из списка; не скачана —
просто пропустит. Первый прогон каждой модели включает её загрузку в память.


In [ ]:
MODELS_TO_COMPARE = ["qwen2.5:3b", "qwen2.5:1.5b", "qwen2.5:0.5b-base", "qwen3.5:9b"]

reports = []
if OLLAMA_UP:
    installed = {m["name"] for m in httpx.get(f"{OLLAMA_URL}/api/tags", timeout=5).json()["models"]}
    for model in MODELS_TO_COMPARE:
        if model not in installed:
            print(f"{model}: не скачана (ollama pull {model}) — пропускаю")
            continue
        print(f"Прогоняю {len(tasks)} задач через {model}...")
        reports.append(bench(model, tasks))
else:
    print("Пропущено: Ollama не запущена")

In [ ]:
for rep in reports:
    tps = f"{rep['median_tps']:.1f} ток/с" if rep["median_tps"] else "n/a"
    print(f"\n{rep['model']}: общий счёт {rep['score']:.0%}, медианная скорость {tps}")
    for cat, score in rep["by_category"].items():
        print(f"  {cat:<16} {score:.0%}")
    for r in rep["results"]:
        if not r["passed"]:
            print(f"  FAIL {r['id']} (done={r['done_reason']}): {r['answer'][:70]!r}")

Мой прогон (у меня инференс на GPU; **счёт от этого не зависит** — модель та же,
меняется только скорость):

| модель | счёт | медианная скорость | done_reason |
|---|---|---|---|
| qwen2.5:3b | **10/10** | ~91 ток/с | у всех `stop` |
| qwen2.5:0.5b-base | **2/10** | ~281 ток/с | у ВСЕХ `length` |

Как это читать:

- **3b прошла всё.** На GPU `temperature=0` не даёт бит-в-бит повторяемости
  (мы разбирали это в уроке 3.3), так что у вас счёт может гулять на ±1 задачу —
  но классификация и извлечение у 3B стабильны.
- **0.5b-base провалилась не потому, что маленькая, а потому, что base.**
  Она не отвечает на вопрос, а *продолжает текст*: в её ответах — эхо задания
  и обрывки чужого chat-шаблона («You are a helpful assistant»), а `done_reason`
  у всех задач — `length`: модель ни разу не остановилась сама, её обрезал
  `num_predict`. Это живая иллюстрация урока 1.6: base-модель — автодополнялка,
  диагноз ставится по `done_reason`.
- **Мягкие чекеры пропустили 2 сломанных ответа**: `contains_all` нашёл
  «SteelTrack» внутри бессвязного продолжения, а `check_json_keys` вырезал валидный
  `{...}` из мусора. Мораль: чем строже чекер (`exact`, `regex`), тем честнее
  бенчмарк — «умная» вырезалка JSON нужна в проде, но в тестах она маскирует брак.
- **281 ток/с у 0.5b — бесполезная скорость.** Скорость без качества не метрика:
  сначала фильтр по счёту, потом сравнение скоростей.

**Правило выбора: берите самую маленькую модель, которая проходит ВАШ бенчмарк
с нужным запасом** (например, ≥90% на критичных категориях). Меньше модель —
быстрее ответ, меньше RAM, дешевле сервер. Проверить, достаточно ли `qwen2.5:1.5b`,
— упражнение 1.

Тот же бенчмарк оформлен как CLI-скрипт `scripts/bench.py` — удобно гонять
из терминала при каждом обновлении модели:

```powershell
python scripts/bench.py --models qwen2.5:3b,qwen2.5:0.5b-base --verbose
```

### Сравнение квантизаций одной модели

В Ollama квантизация зашита в тег: `ollama pull qwen2.5:3b-instruct-q8_0` скачает
Q8_0-вариант (полный список тегов — на странице модели на ollama.com; наш `qwen2.5:3b`
— это Q4_K_M, видно в `ollama show`). Прогоните бенчмарк на обоих: обычно разница
в качестве в пределах шума, а в скорости — почти двукратная. Это лучший способ
прочувствовать, почему Q4_K_M — дефолт. (Я этот вариант не скачивал — модуль
обходится уже установленными моделями; проверьте сами при желании.)


## ⚠️ Частые ошибки

1. **Судить модель по одному промпту.** «Я спросил, и она ответила глупо» — не измерение. Минимум 10–30 задач с автоматической проверкой, иначе вы выбираете по анекдоту.
2. **Скачивать F16/Q8 «для качества».** На CPU это вдвое-вчетверо медленнее при почти неизмеримом приросте качества. Дефолт — Q4_K_M; Q8 — для эталонных сравнений.
3. **Забывать про KV-кеш.** «Файл 4.9 ГБ, у меня 8 ГБ RAM — влезет!» — а потом контекст 16k добавляет ещё 2 ГБ, и система уходит в swap. Считайте по формуле: файл + KV + runtime + аппетиты ОС.
4. **Сравнивать модели в разных условиях.** Разные system-промпты, температура, num_predict — и вот вы сравниваете не модели, а настройки. Фиксируйте всё, кроме самой модели (как в A/B-тесте).
5. **Брать base-модель вместо instruct.** Base продолжает текст, а не выполняет инструкции — мы видели это живьём выше: 2/10, все ответы оборваны по `length`, внутри — эхо задания и обрывки chat-шаблона.

## Упражнения

1. **Своя категория задач + сосед по размеру.** Добавьте в бенчмарк 2–3 задачи из вашей реальной работы (например, извлечение полей из типового письма или классификация по вашим категориям) и подберите тип проверки. Скачайте `ollama pull qwen2.5:1.5b` и выясните: проходит ли она ваш бенчмарк — и можно ли, по правилу выбора, съехать с 3b на 1.5b?
2. **Подбор квантизации под бюджет RAM.** Напишите функцию `pick_quant(params_b, ram_budget_gb, ctx_tokens)`, которая возвращает самую качественную квантизацию из `BITS_PER_WEIGHT`, влезающую в бюджет (используйте `estimate_ram_gb`), или `None`.
3. **Отчёт о скорости.** Расширьте `bench()` так, чтобы он возвращал ещё p10-скорость (медленные хвосты) и время загрузки модели (`load_duration` из ответа Ollama). Подсказка: p10 = `sorted(speeds)[len(speeds) // 10]`.

Решения — ниже.

## Мини-квиз

<details><summary><b>1. Модель 8B в Q4_K_M занимает на диске ~4.9 ГБ. Сколько примерно RAM ей нужно в работе с контекстом 4k?</b></summary>

Около **6–7 ГБ**: 4.9 ГБ файл + ~0.5 ГБ KV-кеш на 4k контекста + ~0.7 ГБ runtime.
Плюс память самой ОС — на машине с 8 ГБ будет впритык (и это ошибка №3 ниже).
</details>

<details><summary><b>2. Что быстрее на CPU и почему: генерация 100 токенов или обработка промпта в 100 токенов?</b></summary>

**Обработка промпта (prefill)**: она выполняется батчем, тогда как генерация выдаёт
токены строго по одному и на каждый читает все веса из RAM. Но масштаб выигрыша
зависит от железа: на CPU честный prefill быстрее генерации в ~3–4 раза (наш замер:
~50–60 против 15–17 ток/с), на GPU — в десятки раз. И помните про кеш префикса:
повтор уже виденного начала промпта почти бесплатен — и портит наивные замеры.
</details>

<details><summary><b>3. У вас 8 ГБ RAM. Что разумнее: 3B в Q8_0 (~3.4 ГБ) или 8B в Q4_K_M (~4.9 ГБ)?</b></summary>

По качеству обычно лучше **8B Q4_K_M** (правило «большая модель в Q4 бьёт маленькую
в Q8»), но на 8 ГБ RAM она оставит системе меньше 3 ГБ — риск swap, и скорость будет
~вдвое ниже, чем у 3B. Если важна отзывчивость и за машиной параллельно работают —
разумнее 3B, причём **Q4, а не Q8**: выигрыша в качестве почти нет, а читать из RAM
приходится вдвое больше.
</details>

<details><summary><b>4. Почему при бенчмарке ставим temperature=0 и фиксируем seed?</b></summary>

Чтобы прогоны были **воспроизводимыми**: изменение счёта тогда означает изменение
модели или промпта, а не случайность, — бенчмарк превращается в регрессионный тест.
Оговорка из урока 3.3: на GPU даже t=0 не гарантирует бит-в-бит — пограничные
задачи могут мерцать, поэтому смотрите на счёт по категориям, а не на одну задачу.
</details>

<details><summary><b>5. Ollama вернула eval_count=120, eval_duration=15_000_000_000. Какова скорость генерации?</b></summary>

`eval_duration` — в **наносекундах**: 15_000_000_000 нс = 15 с, значит
120 / 15 = **8 токенов/сек** — типичная скорость 8B-модели на ноутбучном CPU.
</details>


## Итоги

- Квантизация — сжатие весов с потерями; GGUF — формат «всё в одном файле»; Q4_K_M — рабочий дефолт для CPU.
- RAM = файл модели + KV-кеш (растёт с контекстом) + ~0.7 ГБ runtime. 8 ГБ хватает на 1–3B, 16 ГБ — на 7–8B.
- Генерация на CPU упирается в пропускную способность RAM. Формула даёт потолок; живой замер показал КПД ~40% от пика (3B → 15–17 ток/с на моём ноутбуке).
- Честный prefill на CPU в ~3–4 раза быстрее генерации (в десятки раз — на GPU), а кеш префикса делает шаблонную часть промпта почти бесплатной — поэтому «длинный вход, короткий выход» остаётся лучшим профилем. Скорость меряйте уникальными промптами.
- Чужие лидерборды не заменяют свой бенчмарк: 10–30 задач с автопроверкой — это полчаса работы и главный инструмент выбора. Наш живой прогон: instruct 10/10 против base 2/10 — `done_reason` ставит диагноз.
- Выбирайте самую маленькую модель, проходящую ваш бенчмарк, — всё остальное это налог на RAM и латентность.

**Дальше**: урок 5.3 — превращаем Ollama из «запустил в терминале» в надёжный сервис.

## Полезные ссылки

- Документация Ollama API (поля ответа, options): <https://github.com/ollama/ollama/blob/main/docs/api.md>
- Спецификация GGUF: <https://github.com/ggml-org/ggml/blob/master/docs/gguf.md>
- Обсуждение качества k-quants в llama.cpp: <https://github.com/ggml-org/llama.cpp/discussions>
- Готовые GGUF-сборки моделей: <https://huggingface.co/bartowski>


---

## Решения упражнений

### Упражнение 1: своя категория задач

Пример двух добавленных задач (подставьте свои реальные тексты):

In [ ]:
my_tasks = [
    {
        "id": "my-priority-1",
        "category": "my_tickets",
        "prompt": ("Определи приоритет тикета: low, medium или high. Ответь одним словом.\n\n"
                   "Тикет: «Прод лежит, клиенты не могут оплатить заказы, теряем деньги каждую минуту!»"),
        "check": "exact",
        "expected": "high",
        "max_tokens": 6,
    },
    {
        "id": "my-fields-1",
        "category": "my_tickets",
        "prompt": ("Извлеки из письма JSON с ключами order_id (строка) и problem (строка).\n\n"
                   "Письмо: «Здравствуйте! Заказ A-10452 пришёл без зарядного устройства.» "
                   "Ответь только JSON."),
        "check": "json_keys",
        "expected": ["order_id", "problem"],
        "max_tokens": 80,
    },
]

if OLLAMA_UP and reports:
    for model in [r["model"] for r in reports]:
        rep = bench(model, my_tasks)
        print(f"{model}: {rep['score']:.0%} на моих задачах")
else:
    print("Пропущено: нужна запущенная Ollama")

### Упражнение 2: подбор квантизации под бюджет

In [ ]:
QUALITY_ORDER = ["Q8_0", "Q6_K", "Q5_K_M", "Q4_K_M", "Q3_K_M", "Q2_K"]  # от лучшего к худшему


def pick_quant(params_b: float, ram_budget_gb: float, ctx_tokens: int = 4096) -> str | None:
    """Самая качественная квантизация, влезающая в бюджет RAM (None — не влезает никакая)."""
    kv = 32.0 if params_b < 2 else 128.0
    for quant in QUALITY_ORDER:
        if estimate_ram_gb(params_b, quant, ctx_tokens, kv_kb_per_token=kv) <= ram_budget_gb:
            return quant
    return None


for params, budget in [(8.0, 6.0), (8.0, 12.0), (3.2, 4.0), (8.0, 4.0)]:
    print(f"{params}B при бюджете {budget} ГБ -> {pick_quant(params, budget)}")
# Обратите внимание на последний случай: 8B не влезает в 4 ГБ ни в какой разумной
# квантизации — правильный ответ «возьмите модель поменьше», и функция честно говорит None.

### Упражнение 3: расширенный отчёт о скорости

In [ ]:
def run_task_v2(model: str, task: dict, timeout: float = 300.0) -> dict:
    payload = {
        "model": model,
        "messages": [{"role": "user", "content": task["prompt"]}],
        "stream": False,
        "think": False,
        "options": {"temperature": 0, "seed": 42, "num_predict": task.get("max_tokens", 200)},
    }
    data = httpx.post(f"{OLLAMA_URL}/api/chat", json=payload, timeout=timeout).raise_for_status().json()
    eval_count, eval_duration = data.get("eval_count", 0), data.get("eval_duration", 0)
    return {
        "id": task["id"],
        "passed": CHECKERS[task["check"]](data["message"]["content"], task["expected"]),
        "tps": eval_count / (eval_duration / NS) if eval_duration else None,
        "load_seconds": data.get("load_duration", 0) / NS,
    }


def bench_v2(model: str, tasks: list[dict]) -> dict:
    results = [run_task_v2(model, t) for t in tasks]
    speeds = sorted(r["tps"] for r in results if r["tps"])
    return {
        "model": model,
        "score": sum(r["passed"] for r in results) / len(results),
        "median_tps": speeds[len(speeds) // 2] if speeds else None,
        "p10_tps": speeds[len(speeds) // 10] if speeds else None,  # медленный хвост
        "first_load_seconds": results[0]["load_seconds"] if results else None,
    }


if OLLAMA_UP and reports:
    rep = bench_v2(reports[0]["model"], tasks[:5])
    print(rep)
else:
    print("Пропущено: нужна запущенная Ollama")